<a href="https://colab.research.google.com/github/prestonmorg/Social-Media-Impact-on-Life/blob/main/notebooks/Social_Media_Impact_NN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

In [2]:
# We can first read in the data from the GitHub repo and create a dataframe.
data_url = "https://raw.githubusercontent.com/prestonmorg/Social-Media-Impact-on-Life/refs/heads/main/data/Social_media_impact_on_life.csv"
df = pd.read_csv(data_url)
df.head()

,Student_ID,Age,Gender,Academic_Level,Primary_Platform,Daily_Usage_Hours,Weekend_Extra_Hours,Device_Type,Sleep_Duration_Hours,Sleep_Quality_Score,Late_Night_Usage,Social_Comparison_Frequency,Perceived_Stress_Score,Mental_Health_Index,Academic_Performance_GPA,Overall_Impact
0,STU_2024000,19,Female,Undergraduate,Snapchat,7.5,1.5,Smartphone,5.9,2,True,Always,20.0,73,2.73,Neutral
1,STU_2024001,19,Female,Undergraduate,Instagram,4.6,1.3,Smartphone,7.9,3,True,Sometimes,17.0,90,3.21,Beneficial
2,STU_2024002,17,Female,High School,TikTok,10.9,1.7,Smartphone,5.2,1,True,Frequently,18.0,70,3.00,Neutral
3,STU_2024003,16,Female,High School,YouTube,6.0,2.3,Tablet,7.6,4,True,Sometimes,15.0,82,3.42,Beneficial
4,STU_2024004,17,Male,High School,LinkedIn,3.3,2.6,Smartphone,7.6,4,True,Never,16.0,84,3.41,Beneficial


In [ ]:
# We will now begin to clean/pre-process our data. We can first drop the
# 'Student_ID' column since it will not be useful for the model we're building.
# Furthermore, we can change the T/F 'Late_Night_Usage' column into a binary
# column, where 1 is True and 0 is False.
df = df.drop(columns=['Student_ID'])
df['Late_Night_Usage'] = df['Late_Night_Usage'].astype(int)

# We then map our output classes into numerical categories.
target_map = {'Beneficial': 0, 'Neutral': 1, 'Negative': 2}
y = df['Overall_Impact'].map(target_map).values

# We list our feature groups and separate them into the numerical columns and the
# categorical columns. This allows us to work with the Pipeline from sklearn.
num_cols = [
    'Age', 'Daily_Usage_Hours', 'Weekend_Extra_Hours', 'Sleep_Duration_Hours',
    'Sleep_Quality_Score', 'Perceived_Stress_Score', 'Mental_Health_Index',
    'Academic_Performance_GPA', 'Late_Night_Usage'
]

cat_cols = [
    'Gender', 'Academic_Level', 'Primary_Platform', 'Device_Type',
    'Social_Comparison_Frequency'
]

# We then split the data into the training and testing. Note that we do this
# before we transform the data with the Pipeline so as to prevent leakage.
X_train, X_test, y_train, y_test = train_test_split(
    df[num_cols + cat_cols], y, test_size=0.2, random_state=42, stratify=y
)

# We can then start building the pre-processing Pipelines.
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

# We then only fit onto the training data and then transform both the training
# and the testing data.
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# We finally extract category counts for PyTorch nn.Embedding layers later on
# when we actually create the model.
cat_encoder = preprocessor.named_transformers_['cat'].named_steps['encoder']
cate_cardinalities = [len(cats) for cats in cat_encoder.categories_]